# NB07d — 관제권 공역 데이터 수집 (VWorld WFS API)

**목적**: VWorld WFS API를 통해 성남시 일대 **관제권(Control Zone)** 데이터를 공식 수집하고, NB09 제약 레이어 스코어링에 사용할 공식 공역 파일을 생성합니다.

| 항목 | 내용 |
|---|---|
| WFS 레이어 | `lt_c_aisctrc` (관제권) |
| API 방식 | VWorld WFS 1.1.0 (`/req/wfs`) |
| 출처 | 국토지리정보원 VWorld 공간정보 서비스 |
| bbox 좌표 순서 | **ymin,xmin,ymax,xmax** (EPSG:4326 WFS 표준) |

> **비고**: 기존에 사용하던 `LT_C_AISPRHC`(비행금지구역) / `LT_C_AISRESC`(비행제한구역)은 이 노트북의 주 대상이 아닙니다.  
> 성남·서울공항 인근 드론 운항 제약은 **관제권**이 핵심 레이어입니다.

---
**필수 조건**: `processed/seongnam_boundary.gpkg` 파일이 존재해야 합니다.  
없으면 `NB01_admin_boundary_clip.ipynb`를 먼저 실행하세요.

## 1. 라이브러리 임포트 및 경로 설정

In [1]:
import warnings
from datetime import datetime
from pathlib import Path

import folium
import geopandas as gpd
import pandas as pd
import requests
from shapely.geometry import shape
from shapely.validation import make_valid

warnings.filterwarnings("ignore")

# ── 경로 설정 ────────────────────────────────────────────────────────────────
# 01_preprocessing/ 내부 또는 프로젝트 루트 어디서 실행해도 동작
BASE = Path.cwd().parent if Path.cwd().name == "01_preprocessing" else Path.cwd()
PROC = BASE / "processed"

print(f"프로젝트 루트 : {BASE}")
print(f"처리 데이터 폴더: {PROC}")

# ── VWorld WFS 설정 ──────────────────────────────────────────────────────────
VWORLD_KEY      = "CA689AF7-5C7C-37D4-8973-372AEA113858"
WFS_ENDPOINT    = "https://api.vworld.kr/req/wfs"
WFS_TYPENAME    = "lt_c_aisctrc"   # 관제권
WFS_CRS         = "EPSG:4326"
WFS_MAX_FEATURES = 1000

COLLECTED_AT = datetime.now().isoformat()

print(f"\nWFS 레이어 : {WFS_TYPENAME} (관제권)")
print(f"수집 시각  : {COLLECTED_AT}")

프로젝트 루트 : C:\Users\jimin\Desktop\1_BITAmin_16기\1_Seongnam_reset
처리 데이터 폴더: C:\Users\jimin\Desktop\1_BITAmin_16기\1_Seongnam_reset\processed

WFS 레이어 : lt_c_aisctrc (관제권)
수집 시각  : 2026-05-10T15:52:37.305684


## 2. 성남시 행정 경계 로드

NB01에서 생성된 `processed/seongnam_boundary.gpkg`를 로드합니다.

In [2]:
BOUNDARY_PATH = PROC / "seongnam_boundary.gpkg"

if not BOUNDARY_PATH.exists():
    raise FileNotFoundError(
        "Run NB01_admin_boundary_clip.ipynb first.\n"
        f"Missing: {BOUNDARY_PATH}"
    )

import fiona
available_layers = fiona.listlayers(str(BOUNDARY_PATH))
print(f"사용 가능한 레이어: {available_layers}")

if "city" in available_layers:
    boundary_raw = gpd.read_file(BOUNDARY_PATH, layer="city")
    print("'city' 레이어 로드")
elif "dong" in available_layers:
    dong = gpd.read_file(BOUNDARY_PATH, layer="dong")
    boundary_raw = dong.dissolve().reset_index(drop=True)
    print("'dong' dissolve → 시 경계 생성")
else:
    boundary_raw = gpd.read_file(BOUNDARY_PATH, layer=available_layers[0])
    boundary_raw = boundary_raw.dissolve().reset_index(drop=True)
    print(f"'{available_layers[0]}' dissolve 사용")

# WFS 요청용: EPSG:4326
boundary_4326 = boundary_raw.to_crs(epsg=4326)
# 면적/교차 계산용: EPSG:5179 (Korean UTM-K, 미터 단위)
boundary_5179 = boundary_raw.to_crs(epsg=5179)

seongnam_area_km2 = boundary_5179.geometry.area.sum() / 1e6
print(f"\n성남시 경계 CRS: {boundary_4326.crs}")
print(f"성남시 면적    : {seongnam_area_km2:.2f} km²")

사용 가능한 레이어: ['dong', 'city', 'dong_5179']
'city' 레이어 로드

성남시 경계 CRS: EPSG:4326
성남시 면적    : 141.40 km²


## 3. WFS 요청 바운딩박스 구성

VWorld WFS 1.1.0 + EPSG:4326 의 bbox 파라미터 좌표 순서는 **위도(y) 먼저**입니다:

```
bbox=ymin,xmin,ymax,xmax
```

일반적인 GIS 관례(xmin,ymin,xmax,ymax)와 **반대**이므로 주의가 필요합니다.

In [3]:
PAD = 0.05  # 약 5 km 패딩 (관제권은 시 경계를 크게 벗어날 수 있음)

xmin, ymin, xmax, ymax = boundary_4326.total_bounds
xmin -= PAD
ymin -= PAD
xmax += PAD
ymax += PAD

# !! WFS EPSG:4326 bbox 순서: ymin,xmin,ymax,xmax (위도 먼저)
wfs_bbox = f"{ymin:.6f},{xmin:.6f},{ymax:.6f},{xmax:.6f}"

print(f"성남시 원본 bounds  : xmin={xmin+PAD:.4f}, ymin={ymin+PAD:.4f}, xmax={xmax-PAD:.4f}, ymax={ymax-PAD:.4f}")
print(f"패딩 추가 후        : xmin={xmin:.4f}, ymin={ymin:.4f}, xmax={xmax:.4f}, ymax={ymax:.4f}")
print(f"\nWFS bbox 파라미터  : {wfs_bbox}")
print(f"  (순서: ymin,xmin,ymax,xmax — EPSG:4326 WFS 표준)")

성남시 원본 bounds  : xmin=127.0277, ymin=37.3334, xmax=127.1958, ymax=37.4748
패딩 추가 후        : xmin=126.9777, ymin=37.2834, xmax=127.2458, ymax=37.5248

WFS bbox 파라미터  : 37.283375,126.977705,37.524809,127.245819
  (순서: ymin,xmin,ymax,xmax — EPSG:4326 WFS 표준)


## 4. VWorld WFS 관제권 데이터 수집

WFS GetFeature 요청으로 `lt_c_aisctrc`(관제권) 레이어를 수집합니다.  
관제권은 전국적으로 30여 개 이므로 페이지네이션은 maxFeatures/startIndex 방식으로 처리합니다.

In [4]:
def fetch_wfs_page(typename, bbox, start_index=0, max_features=1000):
    """
    VWorld WFS GetFeature 요청 (한 페이지).
    Returns (feature_list, total_matched, status_str)
    """
    params = {
        "service":      "WFS",
        "request":      "GetFeature",
        "version":      "1.1.0",
        "typename":     typename,
        "output":       "application/json",
        "srsname":      WFS_CRS,
        "bbox":         bbox,
        "maxFeatures":  max_features,
        "startIndex":   start_index,
        "key":          VWORLD_KEY,
    }

    # 로그용 URL (API 키 마스킹)
    masked = {k: ("***" if k == "key" else v) for k, v in params.items()}
    log_url = requests.Request("GET", WFS_ENDPOINT, params=masked).prepare().url
    print(f"  요청 URL: {log_url}")

    try:
        resp = requests.get(WFS_ENDPOINT, params=params, timeout=30)
    except requests.exceptions.Timeout:
        print("  오류: 요청 시간 초과 (30초)")
        return [], 0, "TIMEOUT"
    except requests.exceptions.RequestException as e:
        print(f"  오류: 네트워크 실패 — {e}")
        return [], 0, "NETWORK_ERROR"

    if resp.status_code != 200:
        print(f"  오류: HTTP {resp.status_code}")
        print(f"  응답 앞부분: {resp.text[:300]}")
        return [], 0, f"HTTP_{resp.status_code}"

    # XML 오류 응답 감지 (WFS는 오류를 XML로 반환할 수 있음)
    ct = resp.headers.get("Content-Type", "")
    if "xml" in ct and "json" not in ct:
        print(f"  경고: XML 응답 수신 (WFS 오류 가능성). 앞부분: {resp.text[:400]}")
        return [], 0, "XML_ERROR"

    try:
        data = resp.json()
    except ValueError:
        print(f"  오류: JSON 파싱 실패. 응답 앞부분: {resp.text[:300]}")
        return [], 0, "JSON_ERROR"

    features = data.get("features", [])
    # WFS 1.1.0 응답의 totalFeatures (없을 수 있음)
    total_matched = data.get("totalFeatures", data.get("numberMatched", len(features)))

    if not features:
        print(f"  알림: 반환된 피처 없음 (startIndex={start_index})")
        return [], total_matched, "EMPTY"

    return features, total_matched, "OK"


print("WFS 요청 함수 정의 완료")

WFS 요청 함수 정의 완료


In [5]:
# ── 페이지네이션으로 전체 관제권 데이터 수집 ─────────────────────────────────
print("=" * 60)
print(f"VWorld WFS 관제권 수집 시작")
print(f"레이어: {WFS_TYPENAME}")
print(f"Seongnam bbox (y,x 순서): {wfs_bbox}")
print("=" * 60)

all_features = []
wfs_status = None
total_matched = 0

# 첫 페이지
print(f"\n[startIndex=0]")
page_feats, total_matched, wfs_status = fetch_wfs_page(
    WFS_TYPENAME, wfs_bbox, start_index=0, max_features=WFS_MAX_FEATURES
)
all_features.extend(page_feats)
print(f"  → 첫 페이지 피처 수: {len(page_feats)} / 전체 예상: {total_matched}")

# 추가 페이지 (관제권은 전국 30여 개이므로 거의 발생하지 않음)
start_idx = WFS_MAX_FEATURES
while wfs_status == "OK" and len(page_feats) == WFS_MAX_FEATURES:
    print(f"\n[startIndex={start_idx}]")
    page_feats, _, ps = fetch_wfs_page(
        WFS_TYPENAME, wfs_bbox, start_index=start_idx, max_features=WFS_MAX_FEATURES
    )
    if ps not in ("OK", "EMPTY") or not page_feats:
        break
    all_features.extend(page_feats)
    print(f"  → 추가 피처: {len(page_feats)}")
    start_idx += WFS_MAX_FEATURES

print(f"\n수집 완료: 총 {len(all_features)}개 피처 (WFS 상태: {wfs_status})")

VWorld WFS 관제권 수집 시작
레이어: lt_c_aisctrc
Seongnam bbox (y,x 순서): 37.283375,126.977705,37.524809,127.245819

[startIndex=0]
  요청 URL: https://api.vworld.kr/req/wfs?service=WFS&request=GetFeature&version=1.1.0&typename=lt_c_aisctrc&output=application%2Fjson&srsname=EPSG%3A4326&bbox=37.283375%2C126.977705%2C37.524809%2C127.245819&maxFeatures=1000&startIndex=0&key=%2A%2A%2A
  → 첫 페이지 피처 수: 3 / 전체 예상: 3

수집 완료: 총 3개 피처 (WFS 상태: OK)


## 5. GeoJSON 피처 → GeoDataFrame 변환

WFS가 반환한 GeoJSON 피처를 파싱하고 CRS를 설정합니다.  
무효 지오메트리는 `make_valid()`로 수정합니다.

In [6]:
def parse_wfs_features(features):
    """WFS GeoJSON 피처 리스트를 GeoDataFrame으로 변환합니다."""
    if not features:
        return gpd.GeoDataFrame()

    rows = []
    for feat in features:
        props = feat.get("properties") or {}
        geom_raw = feat.get("geometry")

        geom = None
        if geom_raw and isinstance(geom_raw, dict) and geom_raw.get("type"):
            try:
                geom = shape(geom_raw)
            except Exception as e:
                print(f"  지오메트리 파싱 실패: {e}")

        row = dict(props)
        row["geometry"] = geom
        rows.append(row)

    gdf = gpd.GeoDataFrame(rows, geometry="geometry")
    # 컬럼명 소문자 통일
    gdf.columns = [c.lower() for c in gdf.columns]
    return gdf


if all_features:
    gdf_raw = parse_wfs_features(all_features)
    gdf_raw = gdf_raw.set_crs(epsg=4326, allow_override=True)

    print(f"원본 GeoDataFrame: {len(gdf_raw)}행 × {len(gdf_raw.columns)}열")
    print(f"CRS: {gdf_raw.crs}")
    print(f"컬럼 목록: {list(gdf_raw.columns)}")
    print(f"\n속성 샘플:")
    display(gdf_raw.drop(columns=["geometry"], errors="ignore").head())
else:
    gdf_raw = gpd.GeoDataFrame()
    print("\n!! 수집된 피처가 없습니다.")
    print(f"   WFS 상태: {wfs_status}")
    print(f"   레이어   : {WFS_TYPENAME} (관제권)")
    print("   가능한 원인: bbox 좌표 순서 오류, API 키 문제, 서비스 일시 중단")

원본 GeoDataFrame: 3행 × 4열
CRS: EPSG:4326
컬럼 목록: ['ctr_label', 'ctr_class_', 'ctr_lbl_1', 'geometry']

속성 샘플:


,ctr_label,ctr_class_,ctr_lbl_1
0,None,None,SEOUL
1,None,None,SEOUL CTLZ
2,None,None,SUWON CTLZ


## 6. 지오메트리 정제 및 메타데이터 추가

빈 지오메트리를 제거하고 무효 폴리곤을 `make_valid()`로 수정합니다.

In [7]:
if not gdf_raw.empty:
    # ── 빈/null 지오메트리 제거 ──────────────────────────────────────────────
    valid_mask = gdf_raw.geometry.notna() & ~gdf_raw.geometry.is_empty
    gdf_valid = gdf_raw[valid_mask].copy()
    dropped = len(gdf_raw) - len(gdf_valid)
    if dropped:
        print(f"빈/null 지오메트리 제거: {dropped}개")

    # ── 무효 지오메트리 수정 ─────────────────────────────────────────────────
    invalid_mask = ~gdf_valid.geometry.is_valid
    if invalid_mask.any():
        n_inv = invalid_mask.sum()
        print(f"무효 지오메트리 수정 중 (make_valid): {n_inv}개")
        gdf_valid.loc[invalid_mask, "geometry"] = (
            gdf_valid.loc[invalid_mask, "geometry"].apply(make_valid)
        )
    else:
        print("모든 지오메트리 유효 ✓")

    # ── CRS 확인 ─────────────────────────────────────────────────────────────
    if gdf_valid.crs is None:
        gdf_valid = gdf_valid.set_crs(epsg=4326)
    elif gdf_valid.crs.to_epsg() != 4326:
        gdf_valid = gdf_valid.to_crs(epsg=4326)

    # ── 메타데이터 컬럼 추가 ─────────────────────────────────────────────────
    gdf_valid["source"]           = "VWorld WFS"
    gdf_valid["source_layer"]     = WFS_TYPENAME
    gdf_valid["airspace_type"]    = "관제권"
    gdf_valid["official_airspace"] = True
    gdf_valid["collected_at"]     = COLLECTED_AT

    print(f"\n유효 지오메트리: {len(gdf_valid)} / {len(gdf_raw)}")
    print(f"CRS: {gdf_valid.crs}")
    print(f"공간 범위: {gdf_valid.total_bounds}")
else:
    gdf_valid = gpd.GeoDataFrame()
    print("원본 피처 없음 → 지오메트리 정제 건너뜀")

모든 지오메트리 유효 ✓

유효 지오메트리: 3 / 3
CRS: EPSG:4326
공간 범위: [126.9028245   37.155972   127.21881085  37.52926627]


## 7. 성남시와의 교차 분석 및 면적 계산

면적 및 교차 계산은 **EPSG:5179** (미터 단위)에서 수행합니다.  
지도 출력용으로는 EPSG:4326 버전도 유지합니다.

In [8]:
gdf_seongnam = gpd.GeoDataFrame()
gdf_seongnam_4326 = gpd.GeoDataFrame()

if not gdf_valid.empty:
    # 5179로 투영 (면적/교차 계산용)
    gdf_valid_5179 = gdf_valid.to_crs(epsg=5179)
    seongnam_union_5179 = (
        boundary_5179.union_all()
        if hasattr(boundary_5179, "union_all")
        else boundary_5179.unary_union
    )

    # 교차 필터
    intersects_mask = gdf_valid_5179.geometry.intersects(seongnam_union_5179)
    gdf_intersect_5179 = gdf_valid_5179[intersects_mask].copy()
    print(f"성남시 교차 피처: {len(gdf_intersect_5179)} / {len(gdf_valid_5179)}")

    if not gdf_intersect_5179.empty:
        # 교차 클리핑 (5179)
        gdf_seongnam = gpd.clip(gdf_intersect_5179, boundary_5179)
        print(f"클리핑 후 피처  : {len(gdf_seongnam)}")

        # 면적 계산 (5179 기준, 미터/km²)
        gdf_seongnam["area_m2"]  = gdf_seongnam.geometry.area
        gdf_seongnam["area_km2"] = gdf_seongnam["area_m2"] / 1e6
        total_cz_area = gdf_seongnam["area_km2"].sum()
        gdf_seongnam["intersection_ratio_vs_seongnam"] = (
            gdf_seongnam["area_km2"] / seongnam_area_km2
        )

        print(f"\n성남시 내 관제권 면적: {total_cz_area:.4f} km²")
        print(f"성남시 전체 면적    : {seongnam_area_km2:.2f} km²")
        print(f"관제권 비율         : {total_cz_area / seongnam_area_km2 * 100:.1f}%")

        # 4326 버전 (지도 출력용)
        gdf_seongnam_4326 = gdf_seongnam.to_crs(epsg=4326)

        print(f"\n관제권 피처 속성:")
        display(gdf_seongnam.drop(columns=["geometry"]).head())
    else:
        print("\n⚠️ 성남시 범위와 교차하는 관제권 피처 없음")
else:
    print("원본 피처 없음 → 교차 분석 건너뜀")

성남시 교차 피처: 2 / 3
클리핑 후 피처  : 2

성남시 내 관제권 면적: 241.3822 km²
성남시 전체 면적    : 141.40 km²
관제권 비율         : 170.7%

관제권 피처 속성:


,ctr_label,ctr_class_,ctr_lbl_1,source,source_layer,airspace_type,official_airspace,collected_at,area_m2,area_km2,intersection_ratio_vs_seongnam
1,None,None,SEOUL CTLZ,VWorld WFS,lt_c_aisctrc,관제권,True,2026-05-10T15:52:37.305684,1.206178e+08,120.617811,0.853043
0,None,None,SEOUL,VWorld WFS,lt_c_aisctrc,관제권,True,2026-05-10T15:52:37.305684,1.207644e+08,120.764393,0.854080


## 8. 출력 파일 저장

| 파일 | 내용 |
|---|---|
| `airspace_control_zone_raw.gpkg` | WFS 원본 전체 피처 |
| `airspace_control_zone_seongnam.gpkg` | 성남시 클리핑 결과 (5179) |
| `airspace_constraint.gpkg` | NB09 호환 공식 공역 파일 |
| `airspace_control_zone_summary.csv` | 속성 요약 |
| `airspace_control_zone_map.html` | Folium 인터랙티브 지도 |

In [9]:
saved_files = []
constraint_path = PROC / "airspace_constraint.gpkg"

# ── 1. 원본 전체 피처 ────────────────────────────────────────────────────────
raw_path = PROC / "airspace_control_zone_raw.gpkg"
if not gdf_valid.empty:
    gdf_valid.to_file(raw_path, layer="control_zone_raw", driver="GPKG")
    print(f"저장: {raw_path.name} ({len(gdf_valid)}개 피처, EPSG:4326)")
    saved_files.append(str(raw_path))
else:
    print(f"원본 피처 없음 → {raw_path.name} 생략")

# ── 2. 성남시 클리핑 결과 ────────────────────────────────────────────────────
seongnam_path = PROC / "airspace_control_zone_seongnam.gpkg"
if not gdf_seongnam.empty:
    gdf_seongnam.to_file(seongnam_path, layer="control_zone_seongnam", driver="GPKG")
    print(f"저장: {seongnam_path.name} ({len(gdf_seongnam)}개 피처, EPSG:5179)")
    saved_files.append(str(seongnam_path))
else:
    print(f"교차 피처 없음 → {seongnam_path.name} 생략")

# ── 3. NB09 호환 airspace_constraint.gpkg ────────────────────────────────────
NB09_COLS = [
    "airspace_type", "source", "source_layer",
    "official_airspace", "area_m2", "area_km2", "geometry",
]

if not gdf_seongnam.empty:
    # 없는 컬럼은 빈 값으로 채움
    for col in NB09_COLS:
        if col not in gdf_seongnam.columns and col != "geometry":
            gdf_seongnam[col] = ""

    gdf_constraint = gdf_seongnam[[c for c in NB09_COLS if c in gdf_seongnam.columns]].copy()
    gdf_constraint.to_file(constraint_path, layer="airspace_constraint", driver="GPKG")
    print(f"저장: {constraint_path.name} ({len(gdf_constraint)}개 피처) ← NB09 사용")
    saved_files.append(str(constraint_path))
else:
    print(f"\n⚠️ 교차 피처 없음 → {constraint_path.name} 미생성")
    print("   공식 관제권 데이터 없이는 airspace_constraint.gpkg를 생성하지 않습니다.")
    gdf_constraint = gpd.GeoDataFrame()

저장: airspace_control_zone_raw.gpkg (3개 피처, EPSG:4326)
저장: airspace_control_zone_seongnam.gpkg (2개 피처, EPSG:5179)
저장: airspace_constraint.gpkg (2개 피처) ← NB09 사용


In [10]:
# ── 4. CSV 요약 ──────────────────────────────────────────────────────────────
summary_path = PROC / "airspace_control_zone_summary.csv"

if not gdf_seongnam.empty:
    summary_df = gdf_seongnam.drop(columns=["geometry"]).copy()
    summary_df.to_csv(summary_path, index=False, encoding="utf-8-sig")
    print(f"저장: {summary_path.name} ({len(summary_df)}행)")
else:
    # 빈 진단 요약 저장 (교차 피처 없는 경우)
    diag_df = pd.DataFrame([{
        "wfs_status":       wfs_status,
        "raw_features":     len(all_features),
        "valid_geoms":      len(gdf_valid),
        "seongnam_intersect": 0,
        "note":             "관제권 데이터가 성남시 범위와 교차하지 않음",
        "collected_at":     COLLECTED_AT,
    }])
    diag_df.to_csv(summary_path, index=False, encoding="utf-8-sig")
    print(f"저장: {summary_path.name} (빈 진단 요약)")

saved_files.append(str(summary_path))
print(f"\n총 저장 파일: {len(saved_files)}개")

저장: airspace_control_zone_summary.csv (2행)

총 저장 파일: 4개


## 9. 진단 통계

In [11]:
print("=" * 60)
print("진단 통계")
print("=" * 60)
print(f"Seongnam bbox (y,x순서): {wfs_bbox}")
print(f"WFS 레이어             : {WFS_TYPENAME} (관제권)")
print(f"WFS 응답 상태          : {wfs_status}")
print(f"원본 피처 수           : {len(all_features)}")
print(f"유효 지오메트리 수     : {len(gdf_valid)}")
print(f"성남시 교차 피처 수    : {len(gdf_seongnam)}")

if not gdf_seongnam.empty:
    total_area = gdf_seongnam["area_km2"].sum()
    print(f"교차 총 면적          : {total_area:.4f} km²")
    print(f"성남시 대비 비율      : {total_area / seongnam_area_km2 * 100:.1f}%")
    print(f"\n관제권 피처 목록:")
    cols_to_show = [c for c in gdf_seongnam.columns
                    if c not in ("geometry",) and not c.startswith("_")]
    display(gdf_seongnam[cols_to_show])
else:
    print(f"교차 총 면적          : 0 km²")

print(f"\nairspace_constraint.gpkg 생성: {'예 ✓' if constraint_path.exists() else '아니오 ✗'}")

진단 통계
Seongnam bbox (y,x순서): 37.283375,126.977705,37.524809,127.245819
WFS 레이어             : lt_c_aisctrc (관제권)
WFS 응답 상태          : OK
원본 피처 수           : 3
유효 지오메트리 수     : 3
성남시 교차 피처 수    : 2
교차 총 면적          : 241.3822 km²
성남시 대비 비율      : 170.7%

관제권 피처 목록:


,ctr_label,ctr_class_,ctr_lbl_1,source,source_layer,airspace_type,official_airspace,collected_at,area_m2,area_km2,intersection_ratio_vs_seongnam
1,None,None,SEOUL CTLZ,VWorld WFS,lt_c_aisctrc,관제권,True,2026-05-10T15:52:37.305684,1.206178e+08,120.617811,0.853043
0,None,None,SEOUL,VWorld WFS,lt_c_aisctrc,관제권,True,2026-05-10T15:52:37.305684,1.207644e+08,120.764393,0.854080



airspace_constraint.gpkg 생성: 예 ✓


## 10. Folium 지도

- **파란 점선**: 성남시 행정 경계
- **파란 실선 (반투명)**: WFS 원본 관제권 전체 폴리곤
- **빨간 채우기**: 성남시와 교차하는 관제권 클리핑 영역

In [12]:
center = boundary_4326.geometry.centroid.iloc[0]
m = folium.Map(
    location=[center.y, center.x],
    zoom_start=11,
    tiles="CartoDB positron"
)

# ── 성남시 경계 ──────────────────────────────────────────────────────────────
folium.GeoJson(
    boundary_4326.__geo_interface__,
    name="성남시 경계",
    style_function=lambda x: {
        "fillColor": "transparent",
        "color": "#2c7bb6",
        "weight": 2.5,
        "dashArray": "6, 4",
    },
    tooltip="성남시 행정 경계"
).add_to(m)

# ── 원본 관제권 폴리곤 (전체 범위, 반투명) ──────────────────────────────────
if not gdf_valid.empty:
    folium.GeoJson(
        gdf_valid.__geo_interface__,
        name="관제권 (원본 전체)",
        style_function=lambda x: {
            "fillColor": "#4292c6",
            "color":     "#08519c",
            "weight":    1.5,
            "fillOpacity": 0.15,
        },
        tooltip=folium.GeoJsonTooltip(
            fields=[c for c in gdf_valid.columns
                    if c not in ("geometry", "collected_at") and not c.startswith("_")][:6],
            aliases=None,
            localize=True,
        )
    ).add_to(m)
    print(f"원본 관제권 레이어: {len(gdf_valid)}개 폴리곤")

# ── 성남시 교차 관제권 (클리핑 결과) ────────────────────────────────────────
if not gdf_seongnam_4326.empty:
    for _, row in gdf_seongnam_4326.iterrows():
        if row.geometry is None or row.geometry.is_empty:
            continue
        tooltip_html = (
            f"<b>[공식] 관제권</b><br>"
            f"레이어: {row.get('source_layer', WFS_TYPENAME)}<br>"
            f"면적: {row.get('area_km2', 0):.3f} km²<br>"
            f"성남시 비율: {row.get('intersection_ratio_vs_seongnam', 0)*100:.1f}%"
        )
        folium.GeoJson(
            row.geometry.__geo_interface__,
            style_function=lambda x: {
                "fillColor": "#d73027",
                "color":     "#a50026",
                "weight":    2,
                "fillOpacity": 0.40,
            },
            tooltip=folium.Tooltip(tooltip_html)
        ).add_to(m)
    print(f"성남시 교차 관제권 레이어: {len(gdf_seongnam_4326)}개 폴리곤")

# ── 범례 ─────────────────────────────────────────────────────────────────────
legend_html = """
<div style="position:fixed;bottom:30px;left:30px;z-index:1000;
     background:white;padding:14px;border-radius:8px;
     border:2px solid #555;font-size:13px;font-family:sans-serif;">
  <b>관제권 (Control Zone)</b><br>
  <span style="border:2px dashed #2c7bb6;width:16px;height:0;display:inline-block;
               margin-right:6px;vertical-align:middle;"></span>성남시 경계<br>
  <span style="background:#4292c6;opacity:0.4;width:14px;height:14px;display:inline-block;
               margin-right:6px;"></span>관제권 원본 (전체)<br>
  <span style="background:#d73027;width:14px;height:14px;display:inline-block;
               margin-right:6px;"></span>관제권 ∩ 성남시 (공식)
</div>
"""
m.get_root().html.add_child(folium.Element(legend_html))
folium.LayerControl().add_to(m)

map_path = PROC / "airspace_control_zone_map.html"
m.save(str(map_path))
saved_files.append(str(map_path))
print(f"\n지도 저장: {map_path.name}")
m

원본 관제권 레이어: 3개 폴리곤
성남시 교차 관제권 레이어: 2개 폴리곤

지도 저장: airspace_control_zone_map.html


## 11. 최종 검증 및 다음 단계 안내

In [13]:
print("=" * 60)
print("최종 검증")
print("=" * 60)

if not all_features:
    print("\n⚠️  WFS 응답에서 피처를 가져오지 못했습니다.")
    print(f"   WFS 상태: {wfs_status}")
    print("   확인 항목:")
    print("     1. API 키 유효 여부")
    print("     2. bbox 좌표 순서 (ymin,xmin,ymax,xmax 확인)")
    print(f"        현재 bbox: {wfs_bbox}")
    print("     3. VWorld WFS 서비스 상태: https://www.vworld.kr")
    print("     4. typename 철자: lt_c_aisctrc (소문자)")
    print("\n   ※ 공식 데이터 없이는 airspace_constraint.gpkg를 생성하지 않습니다.")

elif gdf_seongnam.empty:
    print(f"\n⚠️  관제권 피처 {len(gdf_valid)}개 수집됨, 하지만 성남시와 교차하지 않음.")
    print("   원인 가능성:")
    print("     - bbox 패딩이 충분하지 않을 수 있음")
    print("     - 관제권이 성남시를 완전히 포함하여 경계만 밖에 있을 수 있음")
    print(f"   원본 공간 범위: {gdf_valid.total_bounds}")
    print(f"   성남시 범위   : {boundary_4326.total_bounds}")
    print("\n   → airspace_control_zone_raw.gpkg 저장됨 (진단용)")
    print("   → airspace_constraint.gpkg 미생성")
    print("   ※ 공식 데이터 없이는 NB09 재실행이 필요하지 않습니다.")

elif constraint_path.exists():
    total_area = gdf_seongnam["area_km2"].sum()
    ratio_pct  = total_area / seongnam_area_km2 * 100
    print("\n✅  NB09 can now use official VWorld 관제권 data.")
    print(f"   airspace_constraint.gpkg: {len(gdf_constraint)}개 피처")
    print(f"   성남시 내 관제권 면적   : {total_area:.4f} km² ({ratio_pct:.1f}%)")
    print(f"   레이어                  : {WFS_TYPENAME}")
    print(f"   official_airspace       : True")

# ── 저장 파일 목록 ───────────────────────────────────────────────────────────
print(f"\n저장된 파일 ({len(saved_files)}개):")
for f in saved_files:
    p = Path(f)
    size_kb = p.stat().st_size / 1024 if p.exists() else 0
    print(f"  {p.name:52s} {size_kb:8.1f} KB")

# ── 다음 단계 ────────────────────────────────────────────────────────────────
if constraint_path.exists():
    print("\n" + "=" * 60)
    print("다음 단계: 아래 노트북을 순서대로 재실행하세요")
    print("=" * 60)
    print("  1. 02_analysis/NB09_constraint_layer_scoring.ipynb")
    print("     → airspace_constraint.gpkg를 공식 관제권 소스로 사용")
    print("  2. 02_analysis/NB09b_delivery_zone_classification.ipynb")
    print("     → 갱신된 제약 스코어로 배달 구역 재분류")
    print("  3. 02_analysis/NB10_candidate_site_scoring.ipynb")
    print("     → 최종 허브 후보지 스코어 갱신")
else:
    print("\nairspace_constraint.gpkg 미생성 → NB09 재실행 불필요")
    print("NB09는 기존 서울공항 프록시 모드를 유지합니다.")

최종 검증

✅  NB09 can now use official VWorld 관제권 data.
   airspace_constraint.gpkg: 2개 피처
   성남시 내 관제권 면적   : 241.3822 km² (170.7%)
   레이어                  : lt_c_aisctrc
   official_airspace       : True

저장된 파일 (5개):
  airspace_control_zone_raw.gpkg                          100.0 KB
  airspace_control_zone_seongnam.gpkg                     168.0 KB
  airspace_constraint.gpkg                                168.0 KB
  airspace_control_zone_summary.csv                         0.4 KB
  airspace_control_zone_map.html                          369.7 KB

다음 단계: 아래 노트북을 순서대로 재실행하세요
  1. 02_analysis/NB09_constraint_layer_scoring.ipynb
     → airspace_constraint.gpkg를 공식 관제권 소스로 사용
  2. 02_analysis/NB09b_delivery_zone_classification.ipynb
     → 갱신된 제약 스코어로 배달 구역 재분류
  3. 02_analysis/NB10_candidate_site_scoring.ipynb
     → 최종 허브 후보지 스코어 갱신
